### Compute power, sensitivity and PPV for results obtained over groups of datasets

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)
# display(dict_candidates)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
#path_unfair_dataset = './experiments/unfair_dataset.pkl'
#with open(path_unfair_dataset, "rb") as f:
#    true_unfair_objs_ids = pickle.load(f)['obj_ids']

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Compute the number of objects associated with each candidate.
flattened_list_candidates = dict_candidates['flat_ids']
start_pos_candidates = dict_candidates['start_pos']
num_objs_candidates = np.diff(start_pos_candidates)

Now process the results...

In [ ]:
path_unfair_datasets = './experiments/num_objects/'
list_files_datasets = [f for f in Path(path_unfair_datasets).iterdir() if (f.is_file() and 'results' not in f.name)]
list_files_results = [f for f in Path(path_unfair_datasets).iterdir() if (f.is_file() and 'results' in f.name)]

# Read the results computed over a given group of datasets from disk.
idx_tuple_list_objs = 1
for path_results, path_datasets in zip(list_files_results, list_files_datasets):
    
    # Read the results computed over a group of datasets.
    with open(path_results, "rb") as f:
        set_results = pickle.load(f)
    # display(set_results)

    # Compute the number of datasets in this group.
    num_datasets_group = len(set_results['idx_candidates'])

    # 1 - Determine the statistical power of the approach considering all the candidates (and thus grids) used.
    stat_power = np.sum([len(result) != 0 for result in set_results['idx_candidates']]) / num_datasets_group
    print(stat_power)

    # Read the unfair datasets (needed to retrieve the list of object IDs belonging to the unfair hotspots).
    with open(path_datasets, "rb") as f:
        set_datasets = pickle.load(f)

    sum_sensitivity, sum_ppv = 0., 0.
    for idx_dataset in tqdm(range(num_datasets_group)) :

        # Retrieve the lists of object IDs associated with the various hotspots in this unfair dataset.
        # Each hotspot's list is a 1D numpy array, so we need to concatenate these arrays.
        # Guarantee also that the IDs in the final list are unique.
        set_unfair_obj_ids = np.unique(np.concatenate(set_datasets['data'][idx_dataset][idx_tuple_list_objs]), sorted=False)
        print(set_unfair_obj_ids)

        # Retrieve the extreme candidates detected for this dataset.
        set_detected_candidates = set_results['idx_candidates'][idx_dataset]
        print(set_detected_candidates)

        # TODO: recuperare la lista di oggetti per i candidati "estremi" che sono stati ritrovati per
        #       il dataset correntemente considerato.
        break
    break